# Time Series Data Generation with TimeGAN
### Household Electric Power Consumption (UCI / Kaggle) — MSc Data Analytics, Generative AI Capstone

**Goal.** Train TimeGAN on a real household electricity time series, generate synthetic sequences, and measure how closely they resemble the real data (fidelity) and whether they are useful for a prediction task (utility).

**Pipeline (each step below has a text cell explaining it):**

| # | Step | # | Step |
|---|------|---|------|
| 1 | Setup and configuration | 8 | TimeGAN architecture |
| 2 | Dataset selection and loading | 9 | TimeGAN training (3 phases) |
| 3 | Raw data exploration | 10 | Synthetic data generation and inverse scaling |
| 4 | Data cleaning and hourly resampling | 11 | Visual evaluation (sequences, distributions, PCA, t-SNE) |
| 5 | Exploratory analysis of the cleaned series | 12 | Statistical and temporal evaluation |
| 6 | Chronological train/test split and scaling | 13 | Discriminative score |
| 7 | Sliding-window creation | 14 | Predictive score (TSTR vs TRTR) |
| | | 15 | Results summary, discussion and saving outputs |

> **How to run:** open in Google Colab, set **Runtime → Change runtime type → GPU (T4)**, then **Runtime → Run all**.
> With the default settings training takes roughly 15-30 minutes on a free T4. Lower `ITERATIONS` in Step 1 for a quick trial run.

## Step 1 — Setup and configuration

**What we do:** install the one extra package we need (`kagglehub`, to download the dataset), import the libraries, fix random seeds, and put every tunable setting in one cell.

**Why:** fixed seeds make results reproducible, and keeping hyper-parameters in one place means you can change them without hunting through the notebook.

**Key settings**
- `SEQ_LEN = 24` — each training sample is 24 consecutive hours (one day). This is the window length used for the Stocks and Energy datasets in the TimeGAN paper.
- `HIDDEN_DIM = 24`, `NUM_LAYERS = 3`, `BATCH_SIZE = 128`, `GAMMA = 1` — the defaults from the paper / official code.
- `ITERATIONS` — training steps *per phase*. The official code uses 10,000; 5,000 is a good compromise for a one-month project.
- `MAX_TRAIN_WINDOWS` — hourly data over four years gives ~27,000 heavily overlapping windows. A random 10,000 is plenty and keeps training fast.

In [ ]:
%pip install -q kagglehub

In [ ]:
import os, json, random, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)

# ---------------- configuration ----------------
DATA_PATH         = "household_power_consumption.txt"   # looked up first; downloaded from Kaggle if absent
RESAMPLE_RULE     = "1h"        # minute data -> hourly
SEQ_LEN           = 24          # window length (hours)
TEST_FRAC         = 0.20        # last 20 % of the timeline is held out
MAX_TRAIN_WINDOWS = 10000       # cap on windows used to train TimeGAN

HIDDEN_DIM  = 24
NUM_LAYERS  = 3
BATCH_SIZE  = 128
ITERATIONS  = 5000              # training steps per phase (official code: 10000)
GAMMA       = 1.0
LEARNING_RATE = 1e-3

EVAL_EPOCHS = 30                # epochs for the evaluation LSTMs
N_EVAL_RUNS = 3                 # repeat the two scores and report mean +- std

OUT_DIR = "outputs"; os.makedirs(OUT_DIR, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")
print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

## Step 2 — Dataset selection and loading

**Dataset:** *Individual Household Electric Power Consumption* (UCI Machine Learning Repository, mirrored on Kaggle as `uciml/electric-power-consumption-data-set`).

- One house near Paris, France, measured **every minute from December 2006 to November 2010** (about 2.07 million rows, 47 months).
- 7 numeric measurements: `Global_active_power` (kW), `Global_reactive_power` (kVAr), `Voltage` (V), `Global_intensity` (A), and three sub-meters in Wh — `Sub_metering_1` (kitchen), `Sub_metering_2` (laundry room), `Sub_metering_3` (water heater and air-conditioning).
- About 1.25 % of the rows are missing, marked with `?`.

**Why this dataset:** it matches the household-energy domain in the project proposal, it is real sensor data, it has strong daily and weekly cycles for TimeGAN to learn, and its correlated variables make the evaluation interesting.

**What the cell does:** looks for the file locally, then tries to download it from Kaggle with `kagglehub`, and finally falls back to a manual upload dialog (Colab). Missing values (`?`) are read straight in as `NaN`.

In [ ]:
def locate_dataset():
    candidates = [DATA_PATH, "/content/household_power_consumption.txt",
                  "/kaggle/input/electric-power-consumption-data-set/household_power_consumption.txt"]
    for p in candidates:
        if os.path.exists(p):
            return p
    try:
        import kagglehub
        folder = kagglehub.dataset_download("uciml/electric-power-consumption-data-set")
        for root, _, files in os.walk(folder):
            for f in files:
                if f.endswith((".txt", ".csv")):
                    return os.path.join(root, f)
    except Exception as e:
        print("kagglehub download failed:", e)
    try:
        from google.colab import files          # manual upload fallback
        print("Please upload household_power_consumption.txt (download it from the Kaggle page first)")
        return list(files.upload().keys())[0]
    except Exception:
        raise FileNotFoundError("Place household_power_consumption.txt next to this notebook.")

DATA_FILE = locate_dataset()
raw = pd.read_csv(DATA_FILE, sep=";", na_values=["?"], low_memory=False)
print("Loaded:", DATA_FILE, "->", raw.shape)
raw.head()

## Step 3 — Raw data exploration

**What we do:** inspect the structure, data types, missing values and basic statistics *before* changing anything.

**Why:** you need to know what is wrong with the raw data to justify each cleaning decision in your report.

**What to look for**
- `Date` and `Time` are separate text columns, so they must be combined into a real timestamp.
- The missing-value counts should be roughly equal across the seven measurement columns, because when the meter fails it loses *all* fields at once.
- The value ranges show why scaling is needed: `Voltage` is around 240 while `Global_reactive_power` is around 0.1.

In [ ]:
print("Shape:", raw.shape)
print("\nData types:\n", raw.dtypes)
miss = raw.isna().sum().to_frame("missing")
miss["percent"] = (100 * miss["missing"] / len(raw)).round(2)
print("\nMissing values in the raw per-minute data:\n", miss)
raw.describe().T.round(3)

## Step 4 — Data cleaning and hourly resampling

**What we do**
1. Combine `Date` + `Time` into one `datetime` index and sort chronologically.
2. Drop duplicate timestamps (if any).
3. **Resample from per-minute to hourly means.**
4. Fill the remaining gaps by time-based linear interpolation.

**Why**
- TimeGAN needs a regular, gap-free series. A per-minute series of 2 million points makes a 24-step window cover only 24 minutes, which has no interesting daily pattern, and would be far too slow to train on. Hourly data makes a window of 24 steps equal to one full day.
- Averaging 60 readings into one hour smooths sensor noise and shrinks the data from 2.07 million rows to about 34,600.
- Most missing minutes sit in a few long outages. After hourly averaging, only hours that were *entirely* missing remain as gaps, and these are filled by interpolation. We print how many there are so you can report it.

In [ ]:
df = raw.copy()
df["datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"], format="%d/%m/%Y %H:%M:%S")
df = df.drop(columns=["Date", "Time"]).set_index("datetime").sort_index()
df = df[~df.index.duplicated(keep="first")]

# longest outage in the raw minute data
isna = df["Global_active_power"].isna()
runs = isna.ne(isna.shift()).cumsum()
print(f"Longest run of missing minutes: {int(isna.groupby(runs).sum().max())} "
      f"(= {isna.groupby(runs).sum().max()/60:.1f} hours)")

hourly = df.resample(RESAMPLE_RULE).mean()
print("Hourly rows:", len(hourly), "| hours still missing after resampling:",
      int(hourly.isna().any(axis=1).sum()))

hourly = hourly.interpolate(method="time", limit_direction="both")
assert hourly.isna().sum().sum() == 0
FEATURES = list(hourly.columns)
F = len(FEATURES)
print(f"Clean hourly dataset: {hourly.shape}  ({hourly.index.min()}  ->  {hourly.index.max()})")
hourly.head()

## Step 5 — Exploratory analysis of the cleaned series

**What we do:** plot (a) two weeks of consumption, (b) the average consumption for each hour of the day, (c) the monthly average, and (d) the correlation between the seven variables.

**Why:** these plots show the structure TimeGAN has to learn, and you will compare the synthetic data against the same structure later.

**What to look for**
- **Daily cycle:** low consumption at night, peaks in the morning and evening.
- **Seasonality:** higher use in winter, a clear dip in summer (and August holidays).
- **Correlations:** `Global_active_power` and `Global_intensity` are almost perfectly correlated, and the sub-meters add up to part of the total. TimeGAN should reproduce these relationships between features, not just each feature alone.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
hourly["Global_active_power"].iloc[:24*14].plot(ax=ax[0], lw=1)
ax[0].set(title="Two weeks of hourly Global_active_power", ylabel="kW", xlabel="")
hourly.groupby(hourly.index.hour)["Global_active_power"].mean().plot(ax=ax[1], marker="o")
ax[1].set(title="Average by hour of day", xlabel="hour", ylabel="kW")
hourly["Global_active_power"].resample("MS").mean().plot(ax=ax[2])
ax[2].set(title="Monthly average (seasonality)", xlabel="", ylabel="kW")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/eda_patterns.png", dpi=150); plt.show()

plt.figure(figsize=(7, 5.5))
sns.heatmap(hourly.corr(), annot=True, fmt=".2f", cmap="vlag", center=0, cbar=False)
plt.title("Correlation between features (hourly data)")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/eda_correlation.png", dpi=150); plt.show()

## Step 6 — Chronological train/test split and scaling

**What we do**
1. Split the timeline: the **first 80 %** is training data, the **last 20 %** is a held-out test set.
2. Fit a `MinMaxScaler` on the *training part only* and use it to scale both parts to the range [0, 1].

**Why**
- **Chronological split:** this is a time series, so the test set must come from the *future* of the training data. A random split would put almost identical neighbouring hours in both sets (leakage) and make every score look better than it really is.
- **Scaling to [0, 1]:** TimeGAN's networks use sigmoid outputs, so the data must live in [0, 1]. It also stops large-valued columns like `Voltage` dominating the loss.
- **Fit on train only:** computing min/max over the test set would leak information from the future. A few test values may fall slightly outside [0, 1]; we clip them.

In [ ]:
values = hourly.values.astype("float32")
n_train = int(len(values) * (1 - TEST_FRAC))
train_raw, test_raw = values[:n_train], values[n_train:]
print(f"Train: {len(train_raw)} hours  ({hourly.index[0].date()} -> {hourly.index[n_train-1].date()})")
print(f"Test : {len(test_raw)} hours  ({hourly.index[n_train].date()} -> {hourly.index[-1].date()})")

scaler = MinMaxScaler().fit(train_raw)
train_scaled = scaler.transform(train_raw).astype("float32")
test_scaled  = np.clip(scaler.transform(test_raw), 0, 1).astype("float32")
print("Scaled train range:", train_scaled.min(), "to", train_scaled.max())

## Step 7 — Sliding-window creation

**What we do:** cut the scaled series into overlapping windows of `SEQ_LEN = 24` consecutive hours, moving one hour at a time. This produces a 3-D array of shape `(number of windows, 24, 7)`. Training windows are shuffled and capped at `MAX_TRAIN_WINDOWS`.

**Why:** TimeGAN does not model one long series; it learns the distribution of short sequences. Each window is one sample (here, one day of seven measurements). Overlap gives many samples from a limited amount of data. Shuffling is safe *after* windowing because the order inside each window is preserved.

**What to look for:** the printed shapes. Test windows are never shuffled or used for training.

In [ ]:
def make_windows(arr, seq_len):
    w = np.lib.stride_tricks.sliding_window_view(arr, (seq_len, arr.shape[1]))[:, 0]
    return np.ascontiguousarray(w)

train_all = make_windows(train_scaled, SEQ_LEN)
test_win  = make_windows(test_scaled,  SEQ_LEN)

rng = np.random.default_rng(SEED)
order = rng.permutation(len(train_all))[:MAX_TRAIN_WINDOWS]
train_win = train_all[order]

print("All training windows :", train_all.shape)
print("Used to train TimeGAN:", train_win.shape)
print("Held-out test windows:", test_win.shape)

## Step 8 — TimeGAN architecture

**What TimeGAN is:** a GAN for time series that adds an *embedding space* and a *supervised loss*, so it learns the step-by-step dynamics of the data and not only its overall distribution (Yoon, Jarrett & van der Schaar, NeurIPS 2019).

It has **five small recurrent networks** (stacked GRUs, 24 hidden units, 3 layers, as in the paper):

| Network | Maps | Role |
|---|---|---|
| **Embedder** | real data → latent space | compresses each window into a lower-level temporal representation |
| **Recovery** | latent space → data | reconstructs the data from the latent code (autoencoder with the embedder) |
| **Generator** | random noise → latent space | creates *synthetic latent sequences* |
| **Supervisor** | latent step *t* → latent step *t+1* | learns the next-step dynamics; its loss forces the generator to respect temporal order |
| **Discriminator** | latent sequence → real/fake | adversarial signal (bidirectional GRU) |

**Why this is the "existing implementation":** the code below is a compact TensorFlow 2 port of the authors' official implementation (github.com/jsyoon0823/TimeGAN), with the same networks, losses and weights. Writing it out also lets you explain every loss term in your viva.

**What to look for:** the parameter counts printed at the end.

In [ ]:
def make_net(in_dim, hidden, n_layers, out_dim, out_act, name, bidirectional=False):
    inp = tf.keras.Input(shape=(SEQ_LEN, in_dim))
    x = inp
    for _ in range(n_layers):
        gru = tf.keras.layers.GRU(hidden, return_sequences=True)
        x = tf.keras.layers.Bidirectional(gru)(x) if bidirectional else gru(x)
    out = tf.keras.layers.Dense(out_dim, activation=out_act)(x)
    return tf.keras.Model(inp, out, name=name)

H, L = HIDDEN_DIM, NUM_LAYERS
embedder      = make_net(F, H, L,     H, "sigmoid", "embedder")
recovery      = make_net(H, H, L,     F, "sigmoid", "recovery")
generator     = make_net(F, H, L,     H, "sigmoid", "generator")      # input = noise with F channels
supervisor    = make_net(H, H, L - 1, H, "sigmoid", "supervisor")
discriminator = make_net(H, H, L,     1, None,      "discriminator", bidirectional=True)

for m in [embedder, recovery, generator, supervisor, discriminator]:
    print(f"{m.name:14s} parameters: {m.count_params():>7,}")

## Step 9 — TimeGAN training (three phases)

**Phase 1 — Embedding network training.** Train embedder + recovery as an autoencoder (reconstruction loss). The latent space must first be able to represent the data.

**Phase 2 — Supervised loss training.** Train the supervisor on real latent sequences to predict the next latent step. This teaches the model the *temporal dynamics* before any adversarial game starts.

**Phase 3 — Joint training.** All networks train together. In each iteration:
- the **generator + supervisor** take two steps minimising: adversarial loss (fool the discriminator, in both the latent and supervised spaces, weight `GAMMA`) + 100 × √(supervised loss) + 100 × (moment-matching loss on mean and variance);
- the **embedder + recovery** take two steps minimising: 10 × √(reconstruction loss) + 0.1 × supervised loss;
- the **discriminator** takes one step, but only when its loss is above 0.15 so it does not overpower the generator.

**What to look for:** the reconstruction loss in Phase 1 should fall towards ~0.01 or lower and the supervised loss in Phase 2 should fall steadily. In Phase 3 the generator and discriminator losses oscillate by design; they should not run away to zero or to very large values. Training curves are plotted afterwards.

In [ ]:
mse = tf.keras.losses.MeanSquaredError()
def bce(target, logits):
    return tf.reduce_mean(tf.keras.losses.binary_crossentropy(target, logits, from_logits=True))

opt_e0, opt_s = [tf.keras.optimizers.Adam(LEARNING_RATE) for _ in range(2)]
opt_g, opt_e, opt_d = [tf.keras.optimizers.Adam(LEARNING_RATE) for _ in range(3)]

vars_er = embedder.trainable_variables + recovery.trainable_variables
vars_gs = generator.trainable_variables + supervisor.trainable_variables
vars_d  = discriminator.trainable_variables
opt_e0.build(vars_er); opt_s.build(supervisor.trainable_variables)
opt_g.build(vars_gs);  opt_e.build(vars_er); opt_d.build(vars_d)

@tf.function
def step_embed(x):                     # Phase 1
    with tf.GradientTape() as tape:
        x_tilde = recovery(embedder(x))
        recon = mse(x, x_tilde)
        loss = 10.0 * tf.sqrt(recon)
    opt_e0.apply_gradients(zip(tape.gradient(loss, vars_er), vars_er))
    return recon

@tf.function
def step_supervised(x):                # Phase 2
    with tf.GradientTape() as tape:
        h = embedder(x)
        h_sup = supervisor(h)
        loss_s = mse(h[:, 1:, :], h_sup[:, :-1, :])
    v = supervisor.trainable_variables
    opt_s.apply_gradients(zip(tape.gradient(loss_s, v), v))
    return loss_s

@tf.function
def step_generator(x, z):              # Phase 3: generator + supervisor
    with tf.GradientTape() as tape:
        h = embedder(x)
        e_hat = generator(z)
        h_hat = supervisor(e_hat)
        h_sup = supervisor(h)
        x_hat = recovery(h_hat)
        y_fake   = discriminator(h_hat)
        y_fake_e = discriminator(e_hat)
        g_u   = bce(tf.ones_like(y_fake), y_fake)
        g_u_e = bce(tf.ones_like(y_fake_e), y_fake_e)
        g_s   = mse(h[:, 1:, :], h_sup[:, :-1, :])
        # moment matching on mean and standard deviation of the generated data
        m_hat, v_hat = tf.nn.moments(x_hat, [0]); m_r, v_r = tf.nn.moments(x, [0])
        g_v = tf.reduce_mean(tf.abs(tf.sqrt(v_hat + 1e-6) - tf.sqrt(v_r + 1e-6))) \
            + tf.reduce_mean(tf.abs(m_hat - m_r))
        g_loss = g_u + GAMMA * g_u_e + 100.0 * tf.sqrt(g_s) + 100.0 * g_v
    opt_g.apply_gradients(zip(tape.gradient(g_loss, vars_gs), vars_gs))
    return g_loss, g_u, g_s, g_v

@tf.function
def step_embed_joint(x):               # Phase 3: embedder + recovery
    with tf.GradientTape() as tape:
        h = embedder(x)
        recon = mse(x, recovery(h))
        h_sup = supervisor(h)
        g_s = mse(h[:, 1:, :], h_sup[:, :-1, :])
        loss = 10.0 * tf.sqrt(recon) + 0.1 * g_s
    opt_e.apply_gradients(zip(tape.gradient(loss, vars_er), vars_er))
    return recon

def _d_loss(x, z):
    h = embedder(x)
    e_hat = generator(z); h_hat = supervisor(e_hat)
    y_real, y_fake, y_fake_e = discriminator(h), discriminator(h_hat), discriminator(e_hat)
    return bce(tf.ones_like(y_real), y_real) + bce(tf.zeros_like(y_fake), y_fake) \
         + GAMMA * bce(tf.zeros_like(y_fake_e), y_fake_e)

@tf.function
def d_loss_only(x, z):
    return _d_loss(x, z)

@tf.function
def step_discriminator(x, z):
    with tf.GradientTape() as tape:
        loss = _d_loss(x, z)
    opt_d.apply_gradients(zip(tape.gradient(loss, vars_d), vars_d))
    return loss

def get_batch():
    return tf.constant(train_win[np.random.randint(0, len(train_win), BATCH_SIZE)])
def get_noise():
    return tf.random.uniform((BATCH_SIZE, SEQ_LEN, F))

hist = {"phase1_recon": [], "phase2_sup": [], "g_loss": [], "d_loss": [], "g_sup": []}
LOG = max(ITERATIONS // 5, 1)
print("Model ready. Training will run", ITERATIONS, "steps per phase.")

### Phase 1 — Embedding network training
Autoencoder warm-up (embedder + recovery). Printed number = reconstruction MSE.

In [ ]:
t0 = time.time()
for it in range(ITERATIONS):
    hist["phase1_recon"].append(float(step_embed(get_batch())))
    if it % LOG == 0 or it == ITERATIONS - 1:
        print(f"[Phase 1] step {it:5d}/{ITERATIONS}  recon MSE = {hist['phase1_recon'][-1]:.5f}  ({time.time()-t0:.0f}s)")

### Phase 2 — Supervised loss training
The supervisor learns to predict the next latent step. Printed number = supervised MSE.

In [ ]:
t0 = time.time()
for it in range(ITERATIONS):
    hist["phase2_sup"].append(float(step_supervised(get_batch())))
    if it % LOG == 0 or it == ITERATIONS - 1:
        print(f"[Phase 2] step {it:5d}/{ITERATIONS}  supervised MSE = {hist['phase2_sup'][-1]:.5f}  ({time.time()-t0:.0f}s)")

### Phase 3 — Joint training
All five networks train together. This is the longest phase.

In [ ]:
t0 = time.time()
for it in range(ITERATIONS):
    for _ in range(2):                                   # 2 generator + 2 embedder steps per iteration
        x, z = get_batch(), get_noise()
        g_loss, g_u, g_s, g_v = step_generator(x, z)
        step_embed_joint(x)
    x, z = get_batch(), get_noise()
    d_loss = d_loss_only(x, z)
    if float(d_loss) > 0.15:                             # keep the discriminator from overpowering
        d_loss = step_discriminator(x, z)
    hist["g_loss"].append(float(g_loss)); hist["d_loss"].append(float(d_loss)); hist["g_sup"].append(float(g_s))
    if it % LOG == 0 or it == ITERATIONS - 1:
        print(f"[Phase 3] step {it:5d}/{ITERATIONS}  D = {float(d_loss):.3f}  G = {float(g_loss):.3f}  "
              f"G_sup = {float(g_s):.5f}  G_moments = {float(g_v):.4f}  ({time.time()-t0:.0f}s)")

### Training curves
**What to look for:** Phase 1 and 2 curves should drop and flatten. In Phase 3, the discriminator loss should hover (roughly 1-2, not 0), which means neither network is winning outright. Include this figure in your report as evidence that training behaved sensibly.

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(18, 3.6))
for a, k, t in zip(ax, ["phase1_recon", "phase2_sup", "g_loss", "d_loss"],
                   ["Phase 1: reconstruction MSE", "Phase 2: supervised MSE", "Phase 3: generator loss", "Phase 3: discriminator loss"]):
    a.plot(hist[k], lw=.8); a.set_title(t); a.set_xlabel("step")
ax[0].set_yscale("log"); ax[1].set_yscale("log")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/training_curves.png", dpi=150); plt.show()

## Step 10 — Synthetic data generation and inverse scaling

**What we do:** feed random noise through *generator → supervisor → recovery* to produce synthetic windows, then undo the MinMax scaling so the values are back in real units (kW, V, A, Wh).

We generate one synthetic window per real training window, plus an extra batch the size of the test set that is used later for the discriminative test.

**Why:** the scaled version (0-1) is used for the evaluation models, while the original units are used for plots and for your report, because "2.1 kW" means something and "0.18" does not.

**What to look for:** the synthetic values should sit in the same ranges as the real ones, e.g. `Voltage` around 240 V.

In [ ]:
def generate(n):
    out = []
    for i in range(0, n, 1024):
        m = min(1024, n - i)
        z = tf.random.uniform((m, SEQ_LEN, F))
        out.append(recovery(supervisor(generator(z))).numpy())
    return np.concatenate(out).astype("float32")

syn_win      = generate(len(train_win))          # scaled, same size as the real training set
syn_test_win = generate(len(test_win))           # scaled, used for the held-out discriminative test

def to_original_units(w):
    return scaler.inverse_transform(w.reshape(-1, F)).reshape(w.shape)

real_orig = to_original_units(train_win)
syn_orig  = to_original_units(syn_win)
print("Synthetic windows:", syn_win.shape)

pd.concat({"real": pd.DataFrame(real_orig.reshape(-1, F), columns=FEATURES).describe().loc[["mean", "std", "min", "max"]].T,
           "synthetic": pd.DataFrame(syn_orig.reshape(-1, F), columns=FEATURES).describe().loc[["mean", "std", "min", "max"]].T},
          axis=1).round(3)

## Step 11 — Visual evaluation

This step answers: *do the synthetic sequences look like real ones, and do they cover the same variety?*

**(a) Example sequences** — three random real and three random synthetic days for four features. Look for similar shape, scale, and smoothness. Synthetic curves do not need to copy a real day; they need to be *plausible* days.

**(b) Feature distributions** — kernel density plots of each feature, real vs synthetic. Curves that overlap mean the synthetic data matches the real value distribution.

**(c) PCA and t-SNE** — each 24 × 7 window is flattened into one vector and projected to 2-D. This is the diversity test from the TimeGAN paper: if the red (real) and blue (synthetic) clouds overlap, the synthetic data covers the same regions as the real data. If the blue points form a small separate cluster, the model suffers from *mode collapse* (it generates only a few kinds of days).

In [ ]:
show = ["Global_active_power", "Voltage", "Sub_metering_3", "Global_intensity"]
rs = np.random.default_rng(1)
fig, axes = plt.subplots(len(show), 2, figsize=(12, 2.7 * len(show)), sharex=True)
for r, name in enumerate(show):
    f = FEATURES.index(name)
    for c, (data, title) in enumerate([(real_orig, "Real"), (syn_orig, "Synthetic")]):
        for i in rs.choice(len(data), 3, replace=False):
            axes[r, c].plot(data[i, :, f], lw=1.6)
        axes[r, c].set_title(f"{title}: {name}", fontsize=10)
for a in axes[-1]: a.set_xlabel("hour within window")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/sequences.png", dpi=150); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(17, 6)); axes = axes.ravel()
for f, name in enumerate(FEATURES):
    sns.kdeplot(real_orig[:, :, f].ravel(), ax=axes[f], label="Real", color="tab:red", fill=True, alpha=.25)
    sns.kdeplot(syn_orig[:, :, f].ravel(),  ax=axes[f], label="Synthetic", color="tab:blue", fill=True, alpha=.25)
    axes[f].set_title(name, fontsize=10); axes[f].set_xlabel(""); axes[f].set_ylabel("")
axes[0].legend(); axes[-1].axis("off")
plt.suptitle("Feature distributions: real vs synthetic", y=1.01)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/distributions.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
N_VIS = min(1000, len(train_win), len(syn_win))
rv = np.random.default_rng(2)
r = train_win[rv.choice(len(train_win), N_VIS, replace=False)].reshape(N_VIS, -1)
s = syn_win[rv.choice(len(syn_win), N_VIS, replace=False)].reshape(N_VIS, -1)

pca = PCA(n_components=2).fit(r)                     # fit on real data only, project both
pr, ps = pca.transform(r), pca.transform(s)
tsne = TSNE(n_components=2, perplexity=min(40, N_VIS // 3), init="pca", random_state=SEED)
t = tsne.fit_transform(np.vstack([r, s]))

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].scatter(*pr.T, s=8, alpha=.45, c="tab:red", label="Real"); ax[0].scatter(*ps.T, s=8, alpha=.45, c="tab:blue", label="Synthetic")
ax[0].set_title("PCA (fit on real data)"); ax[0].legend()
ax[1].scatter(*t[:N_VIS].T, s=8, alpha=.45, c="tab:red", label="Real"); ax[1].scatter(*t[N_VIS:].T, s=8, alpha=.45, c="tab:blue", label="Synthetic")
ax[1].set_title("t-SNE"); ax[1].legend()
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/pca_tsne.png", dpi=150); plt.show()
print(f"PCA explained variance (2 components): {pca.explained_variance_ratio_.sum():.1%}")

## Step 12 — Statistical and temporal evaluation

**What we do**
- Compare the **mean and standard deviation** of each feature.
- Compare the **correlation matrices** (real vs synthetic) and report the average absolute difference between them. This checks whether relationships *between* features were learned.
- Compare the **autocorrelation** of `Global_active_power` at lags 1-23 hours. This is the most direct check that TimeGAN learned *temporal dynamics*: real consumption is strongly correlated with its recent past.

**What to look for:** similar means/stds, a small mean correlation difference (near 0; values below about 0.1 are good), and an autocorrelation curve for the synthetic data that follows the real one.

In [ ]:
def flat(a): return a.reshape(-1, a.shape[2])
stats = pd.DataFrame({"mean_real": flat(real_orig).mean(0), "mean_syn": flat(syn_orig).mean(0),
                      "std_real":  flat(real_orig).std(0),  "std_syn":  flat(syn_orig).std(0)}, index=FEATURES)
stats["mean_diff_%"] = 100 * (stats.mean_syn - stats.mean_real) / stats.mean_real.abs()
display(stats.round(3))

c_real, c_syn = np.corrcoef(flat(train_win), rowvar=False), np.corrcoef(flat(syn_win), rowvar=False)
corr_gap = float(np.nanmean(np.abs(c_real - c_syn)))
print(f"Mean absolute difference between correlation matrices: {corr_gap:.4f}")

fig, ax = plt.subplots(1, 3, figsize=(17, 4.8))
for a, c, t in zip(ax[:2], [c_real, c_syn], ["Real correlations", "Synthetic correlations"]):
    sns.heatmap(c, ax=a, vmin=-1, vmax=1, cmap="vlag", annot=True, fmt=".1f", cbar=False,
                xticklabels=FEATURES, yticklabels=FEATURES, annot_kws={"size": 7}); a.set_title(t)

def acf(w, f, max_lag=23):
    x = w[:, :, f]
    return [np.corrcoef(x[:, :-k].ravel(), x[:, k:].ravel())[0, 1] for k in range(1, max_lag + 1)]
f0 = FEATURES.index("Global_active_power")
acf_real, acf_syn = acf(train_win, f0), acf(syn_win, f0)
ax[2].plot(range(1, 24), acf_real, "o-", c="tab:red", label="Real"); ax[2].plot(range(1, 24), acf_syn, "s-", c="tab:blue", label="Synthetic")
ax[2].set(title="Autocorrelation of Global_active_power", xlabel="lag (hours)", ylabel="correlation"); ax[2].legend()
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/correlation_acf.png", dpi=150); plt.show()
acf_gap = float(np.mean(np.abs(np.array(acf_real) - np.array(acf_syn))))
print(f"Mean absolute autocorrelation difference (lags 1-23): {acf_gap:.4f}")

## Step 13 — Discriminative score (fidelity)

**What we do:** train a *new*, independent 2-layer LSTM classifier to tell real windows from synthetic ones, then test it on **unseen** data: real windows from the held-out future period vs fresh synthetic windows.

**Score = |0.5 − test accuracy|.** If the synthetic data is indistinguishable from real data, the classifier can do no better than guessing (accuracy 0.5, score 0). A score near 0.5 means the classifier separates them perfectly.

**Lower is better.** Reference points from the TimeGAN paper (different datasets, so only a rough guide): TimeGAN 0.011 on Stocks and 0.236 on Energy, versus 0.1 to 0.5 for older GAN baselines.

**Caveat to mention in the report:** the real test windows come from a later period than the training windows, so a small part of the score reflects normal drift in household behaviour over time, not only generator quality. The score is repeated `N_EVAL_RUNS` times with different seeds and reported as mean ± std.

In [ ]:
def build_classifier(units=32):
    return tf.keras.Sequential([
        tf.keras.Input((SEQ_LEN, F)),
        tf.keras.layers.LSTM(units, return_sequences=True),
        tf.keras.layers.LSTM(units),
        tf.keras.layers.Dense(1, activation="sigmoid")])

early = lambda: tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)

def discriminative_score(real_tr, syn_tr, real_te, syn_te, seed):
    tf.keras.utils.set_random_seed(seed)
    n = min(len(real_tr), len(syn_tr))
    X = np.concatenate([real_tr[:n], syn_tr[:n]]); y = np.r_[np.ones(n), np.zeros(n)]
    Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)
    clf = build_classifier(); clf.compile("adam", "binary_crossentropy", metrics=["accuracy"])
    clf.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=EVAL_EPOCHS, batch_size=128, verbose=0, callbacks=[early()])
    k = min(len(real_te), len(syn_te))
    Xte = np.concatenate([real_te[:k], syn_te[:k]]); yte = np.r_[np.ones(k), np.zeros(k)]
    acc = clf.evaluate(Xte, yte, verbose=0)[1]
    return abs(0.5 - acc), acc

disc = [discriminative_score(train_win, syn_win, test_win, syn_test_win, SEED + i) for i in range(N_EVAL_RUNS)]
disc_scores = np.array([d[0] for d in disc]); disc_acc = np.array([d[1] for d in disc])
print(f"Classifier accuracy on unseen data: {disc_acc.mean():.3f} +- {disc_acc.std():.3f}")
print(f"Discriminative score |0.5 - acc|  : {disc_scores.mean():.3f} +- {disc_scores.std():.3f}   (lower is better)")

## Step 14 — Predictive score (usefulness, "Train on Synthetic, Test on Real")

**What we do:** train a small LSTM to predict the **next hour's `Global_active_power`** from the previous hours of all seven features, in two ways, and test *both* on the same real held-out test windows:

- **TSTR** — trained on **synthetic** data only (the question we care about).
- **TRTR** — trained on **real** data (the upper-bound baseline).

We also report a **persistence baseline** (predict that next hour equals the current hour) so the numbers have context.

**Score = mean absolute error (MAE), lower is better.** Errors are given in the scaled 0-1 range and converted to kW.

**How to read it:** if TSTR is close to TRTR, the synthetic data carries almost the same predictive information as the real data, so it could replace real data for training a forecaster. If TSTR is much worse, the synthetic data is missing important dynamics.

In [ ]:
def build_forecaster(units=32):
    return tf.keras.Sequential([
        tf.keras.Input((SEQ_LEN - 1, F)),
        tf.keras.layers.LSTM(units, return_sequences=True),
        tf.keras.layers.Dense(1, activation="sigmoid")])

def train_and_test(train_data, test_data, seed):
    tf.keras.utils.set_random_seed(seed)
    m = build_forecaster(); m.compile("adam", "mae")
    m.fit(train_data[:, :-1], train_data[:, 1:, :1], validation_split=0.1, epochs=EVAL_EPOCHS,
          batch_size=128, verbose=0, callbacks=[early()])
    return m.evaluate(test_data[:, :-1], test_data[:, 1:, :1], verbose=0)   # MAE in scaled units

# Global_active_power is column 0 of the data, so it is the prediction target
assert FEATURES[0] == "Global_active_power"
kw = float(scaler.data_range_[0])                    # scaled MAE x range = MAE in kW

tstr = np.array([train_and_test(syn_win,   test_win, SEED + i) for i in range(N_EVAL_RUNS)]) * kw
trtr = np.array([train_and_test(train_win, test_win, SEED + i) for i in range(N_EVAL_RUNS)]) * kw
persist = float(np.mean(np.abs(test_win[:, 1:, 0] - test_win[:, :-1, 0]))) * kw

print(f"TRTR (train real, test real)       : {trtr.mean():.4f} +- {trtr.std():.4f} kW")
print(f"TSTR (train synthetic, test real)  : {tstr.mean():.4f} +- {tstr.std():.4f} kW")
print(f"Persistence baseline               : {persist:.4f} kW")
print(f"TSTR / TRTR ratio                  : {tstr.mean() / trtr.mean():.2f}  (1.00 = synthetic as useful as real)")

## Step 15 — Results summary, discussion and saving outputs

**What we do:** collect every metric in one table, plot the predictive comparison, write an automatic plain-language summary of the numbers, and save all outputs (figures, metrics JSON, synthetic data) to the `outputs/` folder.

**How to interpret the final numbers**

| Metric | Measures | Good value |
|---|---|---|
| PCA / t-SNE overlap | diversity (coverage of real data) | clouds overlap, no isolated synthetic cluster |
| Correlation difference | relationships between features | close to 0 (below about 0.1) |
| Autocorrelation difference | temporal dynamics | close to 0 |
| Discriminative score | fidelity | close to 0 (0.5 = trivially separable) |
| Predictive MAE (TSTR vs TRTR) | usefulness | TSTR close to TRTR |

**Limitations to discuss in your report**
1. One house, one country: the findings may not transfer to other households or other domains.
2. Hourly resampling removes minute-level detail (short appliance spikes).
3. 24-hour windows cannot reproduce weekly or seasonal patterns.
4. TimeGAN was trained for a limited number of iterations and one seed; more training and tuning could improve it.
5. The test period is later than the training period, so scores include some natural drift.
6. No formal privacy guarantee: synthetic data *looks* different from the original, but this is not differential privacy.

**Possible extensions:** compare with a simple baseline (e.g. a plain GAN or RCGAN), run an ablation without the supervised loss, or vary `SEQ_LEN`.

In [ ]:
results = {
    "dataset": "Individual household electric power consumption (hourly)",
    "n_hours": int(len(hourly)), "seq_len": SEQ_LEN, "features": FEATURES,
    "n_train_windows": int(len(train_win)), "n_test_windows": int(len(test_win)),
    "iterations_per_phase": ITERATIONS,
    "mean_abs_correlation_gap": corr_gap, "mean_abs_acf_gap": acf_gap,
    "discriminative_score_mean": float(disc_scores.mean()), "discriminative_score_std": float(disc_scores.std()),
    "discriminator_accuracy_mean": float(disc_acc.mean()),
    "predictive_mae_kW": {"TRTR": float(trtr.mean()), "TSTR": float(tstr.mean()), "persistence": persist},
}
summary = pd.DataFrame({
    "Metric": ["Correlation gap (mean abs diff)", "Autocorrelation gap (lags 1-23)", "Discriminative score",
               "Predictive MAE, TRTR (kW)", "Predictive MAE, TSTR (kW)", "Persistence baseline MAE (kW)"],
    "Value": [f"{corr_gap:.4f}", f"{acf_gap:.4f}", f"{disc_scores.mean():.3f} +- {disc_scores.std():.3f}",
              f"{trtr.mean():.4f} +- {trtr.std():.4f}", f"{tstr.mean():.4f} +- {tstr.std():.4f}", f"{persist:.4f}"]})
display(summary)

plt.figure(figsize=(5.5, 3.6))
bars = plt.bar(["TRTR\n(real->real)", "TSTR\n(synthetic->real)", "Persistence"], [trtr.mean(), tstr.mean(), persist],
               yerr=[trtr.std(), tstr.std(), 0], color=["tab:red", "tab:blue", "gray"], capsize=4)
plt.ylabel("MAE (kW), lower is better"); plt.title("Next-hour prediction error on real test data")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/predictive_scores.png", dpi=150); plt.show()

# plain-language summary generated from the numbers above
d = disc_scores.mean()
fid = "very high" if d < 0.1 else "moderate" if d < 0.25 else "limited"
ratio = tstr.mean() / trtr.mean()
use = "almost as useful as" if ratio < 1.25 else "somewhat less useful than" if ratio < 2 else "much less useful than"
print("\nAUTOMATIC SUMMARY")
print(f"- Fidelity is {fid}: a trained classifier reached {disc_acc.mean():.1%} accuracy on unseen data (discriminative score {d:.3f}).")
print(f"- Feature relationships are preserved with a mean correlation gap of {corr_gap:.3f}; temporal structure (autocorrelation gap) differs by {acf_gap:.3f}.")
print(f"- For forecasting, synthetic training data is {use} real data (TSTR {tstr.mean():.3f} kW vs TRTR {trtr.mean():.3f} kW, ratio {ratio:.2f}).")
if trtr.mean() > persist:
    print(f"- NOTE: even the real-data forecaster (TRTR {trtr.mean():.3f} kW) is worse than the persistence baseline ({persist:.3f} kW), "
          "so the TSTR/TRTR comparison is weak evidence. Train longer (raise EVAL_EPOCHS) before drawing conclusions.")

# save everything
json.dump(results, open(f"{OUT_DIR}/results.json", "w"), indent=2)
np.save(f"{OUT_DIR}/synthetic_windows_original_units.npy", syn_orig)
pd.DataFrame(syn_orig[:500].reshape(-1, F), columns=FEATURES).assign(
    window=np.repeat(np.arange(500), SEQ_LEN), hour=np.tile(np.arange(SEQ_LEN), 500)
).to_csv(f"{OUT_DIR}/synthetic_sample_500_windows.csv", index=False)
print("\nSaved to ./" + OUT_DIR + ":", sorted(os.listdir(OUT_DIR)))

In [ ]:
# Optional (Colab): download everything as one zip
try:
    import shutil
    from google.colab import files
    shutil.make_archive("capstone_outputs", "zip", OUT_DIR)
    files.download("capstone_outputs.zip")
except Exception:
    print("Not running in Colab - find the files in the outputs/ folder.")

### References
- Yoon, J., Jarrett, D., & van der Schaar, M. (2019). *Time-series Generative Adversarial Networks.* NeurIPS 2019.
- Esteban, C., Hyland, S. L., & Rätsch, G. (2017). *Real-valued (Medical) Time Series Generation with Recurrent Conditional GANs.* arXiv:1706.02633.
- Hébrail, G., & Bérard, A. (2012). *Individual Household Electric Power Consumption* [Dataset]. UCI Machine Learning Repository.
- van der Maaten, L., & Hinton, G. (2008). *Visualizing Data using t-SNE.* JMLR 9.